# 03 · Statistical Tests: BEV vs. ICE

Two-sided hypothesis tests at $\alpha = 0.05$. Decision rule: $p < 0.05 \implies$ reject $H_0$.

Each test is run on **(a) all ages** and **(b) vehicle age ≤ 5**. Age 5 is the last age where BEVs still have more than 200 vehicle-years of exposure (206, against 119 at age 6). Where the sample allows, **(c)** also controls for vehicle value tier (< ₪75k, ₪75–150k, ₪150k+).

BEV vs. ICE only; Hybrid is excluded. The 2025-01-19 storm is excluded.

**Why the storm is excluded:** the 2025-01-19 storm is a catastrophe event: a rare, one-off tail loss that cannot be rated from a policy's own characteristics. Insurers cover such events through reinsurance and a separate catastrophe loading, not through the baseline pure-premium tariff, so it is left out of the attritional analysis. The claims stay in the data, flagged as `is_cat_event`.

## Setup

In [1]:
suppressWarnings(suppressPackageStartupMessages({
  library(tidyverse)
  library(scales)
}))

alpha  <- 0.05
fmt_p  <- function(p) if_else(p < 0.001, "< 0.001", number(p, accuracy = 0.001))
fmt_ci <- function(lo, hi, acc = 0.01) paste0(number(lo, acc), " – ", number(hi, acc))
decide <- function(p) if_else(p < alpha, "Reject H0", "Fail to reject H0")

# Path to a file in data/processed/; works from notebooks/ or the repository root
processed_path <- function(file) {
  path <- file.path(c("../data/processed", "data/processed"), file) |> keep(file.exists) |> first()
  if (is.na(path)) stop("Cannot find ", file, " (run src/01_clean_datasets.R first)")
  path
}

policies <- read_rds(processed_path("policies_cleaned.rds")) |>
  filter(Powertrain %in% c("BEV", "ICE")) |>
  mutate(Powertrain = factor(as.character(Powertrain), levels = c("BEV", "ICE")),
         value_tier = cut(Vehicle_Value, c(0, 75e3, 150e3, Inf), right = FALSE,
                          labels = c("Low (<75k)", "Medium (75-150k)", "High (150k+)")))

# Non-storm claims inside the policy period (join checked in src/02_verify_join.R)
claims_all <- read_rds(processed_path("claims_cleaned.rds")) |>
  filter(!is_cat_event) |>
  inner_join(select(policies, Policy_ID, Powertrain, Start_Date, End_Date, Vehicle_Age, Vehicle_Value, value_tier),
             by = "Policy_ID", relationship = "many-to-one") |>
  filter(between(Accident_Date, Start_Date, End_Date))

od_claims <- filter(claims_all, Own_Damage_Amount > 0)

freq_data <- policies |>                                                     # Test 1: claims per policy
  left_join(count(od_claims, Policy_ID, name = "n_claims"), by = "Policy_ID") |>
  mutate(n_claims = coalesce(n_claims, 0L))
sev_data <- filter(od_claims, Own_Damage_Amount <= 2 * Vehicle_Value | is.na(Vehicle_Value))   # Test 2: drop entry errors
tl_data  <- filter(od_claims, Claim_Type != "Theft")                         # Test 3: theft is not a repair decision
tp_data  <- filter(claims_all, Claim_Type == "Collision_With_TP", TP_Damage_Amount > 0)        # Test 4

age_cutoff <- 5
levels_ab  <- function(d) list(`(a) All ages` = d, `(b) Age <= 5` = filter(d, Vehicle_Age <= age_cutoff))

In [2]:
# Exact Poisson test of two claim rates: estimate = lambda_BEV / lambda_ICE
rate_test <- function(d) {
  s  <- d |> summarise(claims = sum(n_claims), exposure = sum(exposure), .by = Powertrain) |> arrange(Powertrain)
  pt <- poisson.test(s$claims, s$exposure)
  tibble(BEV = percent(s$claims[1] / s$exposure[1], 0.01), ICE = percent(s$claims[2] / s$exposure[2], 0.01),
         estimate = pt$estimate, ci_low = pt$conf.int[1], ci_high = pt$conf.int[2],
         statistic = paste0("RR = ", number(pt$estimate, 0.01)), p_value = pt$p.value)
}

# Severity: F-test of equal variances, then two-sample t-test (equal variances), both on log cost.
# ratio = exp(mean log BEV - mean log ICE) = ratio of geometric mean costs BEV / ICE
severity_test <- function(d, amount) {
  d  <- mutate(d, cost = {{ amount }})
  ft <- var.test(log(cost) ~ Powertrain, data = d)
  tt <- t.test(log(cost) ~ Powertrain, data = d, var.equal = TRUE)
  tibble(F = ft$statistic, p_F = ft$p.value,
         ratio = exp(tt$estimate[[1]] - tt$estimate[[2]]),
         t = tt$statistic, df = tt$parameter, p_value = tt$p.value,
         statistic = paste0("F = ", number(F, 0.01), " (p = ", number(p_F, 0.001), "); t = ", number(t, 0.01), ", df = ", df))
}

show_severity <- function(res) {
  res |>
    transmute(comparison, F = number(F, 0.01), p_F = fmt_p(p_F), ratio = number(ratio, 0.01),
              t = number(t, 0.01), df, p_value = fmt_p(p_value), decision) |>
    mutate(across(everything(), as.character), across(everything(), ~ replace_na(.x, "-")))
}

# Two-proportion test of the total-loss rate: estimate = pi_BEV - pi_ICE (percentage points)
prop_test <- function(d) {
  s  <- d |> summarise(tl = sum(is_total_loss_clean), n = n(), .by = Powertrain) |> arrange(Powertrain)
  pt <- prop.test(s$tl, s$n)
  tibble(BEV = percent(s$tl[1] / s$n[1], 0.1), ICE = percent(s$tl[2] / s$n[2], 0.1),
         estimate = 100 * (s$tl[1] / s$n[1] - s$tl[2] / s$n[2]),
         ci_low = 100 * pt$conf.int[1], ci_high = 100 * pt$conf.int[2],
         statistic = paste0("X² = ", number(pt$statistic, 0.01)), p_value = pt$p.value)
}

# Run a test on levels (a) and (b), add the decision, format numbers
run <- function(d, test) {
  levels_ab(d) |> map(test) |> list_rbind(names_to = "comparison") |>
    mutate(decision = decide(p_value))
}
show <- function(res, acc = 0.01) {
  res |> mutate(estimate = number(estimate, acc), ci_95 = fmt_ci(ci_low, ci_high, acc),
                p_value = fmt_p(p_value), .keep = "unused", .after = estimate) |>
    select(-any_of(c("ci_low", "ci_high"))) |>
    mutate(across(everything(), as.character), across(everything(), ~ replace_na(.x, "-")))
}

### Sample sizes
Number of observations behind each test, for (a) all ages and (b) vehicle age ≤ 5.

In [3]:
# Observations in each test sample, by comparison level and powertrain
list(`Tests 1, 1b: policies (frequency denominators)`   = freq_data,
     `Test 1: own-damage claims counted`                = od_claims,
     `Test 1b, Test 4: third-party collision claims`    = tp_data,
     `Test 2: own-damage claims (severity)`             = sev_data,
     `Test 3: own-damage claims excl. theft (total loss)` = tl_data) |>
  imap(\(d, name) map(levels_ab(d), \(x) count(x, Powertrain)) |>
         list_rbind(names_to = "comparison") |> mutate(sample = name)) |>
  list_rbind() |>
  mutate(comparison = str_sub(comparison, 1, 3)) |>
  pivot_wider(names_from = c(comparison, Powertrain), values_from = n, names_sep = " ") |>
  mutate(across(-sample, ~ comma(.x, 1)))

sample,(a) BEV,(a) ICE,(b) BEV,(b) ICE
<chr>,<chr>,<chr>,<chr>,<chr>
"Tests 1, 1b: policies (frequency denominators)","4,196","25,850","3,825","10,594"
Test 1: own-damage claims counted,631,"3,712",562,"1,470"
"Test 1b, Test 4: third-party collision claims",251,"1,432",233,556
Test 2: own-damage claims (severity),631,"3,704",562,"1,467"
Test 3: own-damage claims excl. theft (total loss),626,"3,577",557,"1,415"


**Why the sample sizes differ:** each test uses the unit and claims that fit its question. All samples are in-period and exclude the storm.
- **Frequency (Tests 1, 1b):** the unit is the **policy** with its exposure; every policy counts, including those with no claim.
  - **Test 1** counts own-damage claims (`Own_Damage_Amount > 0`).
  - **Test 1b** counts third-party collision claims.
- **Own-damage severity (Test 2):** only claims with a positive cost can enter a log-cost test. The 8 amounts above 2 × vehicle value (entry errors) are also dropped, so this sample is slightly smaller than the claims counted in Test 1.
- **Total loss (Test 3):** own-damage claims without theft, because a stolen car is written off whatever its powertrain.
- **Third-party severity (Test 4):** collisions with a third party and positive third-party damage. These are the same claims counted in Test 1b, including those where the insured car itself had no damage.
- **Age cohort:** the age ≤ 5 columns (b) are subsets of the all-ages columns (a).

## Test 1: Claim Frequency (Exact Poisson Rate Test)
$\lambda$ = own-damage claims per vehicle-year.
- $H_0$: $\lambda_{\text{BEV}} = \lambda_{\text{ICE}}$
- $H_1$: $\lambda_{\text{BEV}} \neq \lambda_{\text{ICE}}$

In [4]:
tier_data <- freq_data |> filter(Vehicle_Age <= age_cutoff, !is.na(value_tier))
t1 <- bind_rows(run(freq_data, rate_test),
                split(tier_data, tier_data$value_tier) |> map(rate_test) |>
                  list_rbind(names_to = "comparison") |>
                  mutate(comparison = paste0("(c) Age <= 5, ", comparison), decision = decide(p_value)))
show(t1)

comparison,BEV,ICE,estimate,ci_95,statistic,p_value,decision
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
(a) All ages,15.85%,15.15%,1.05,0.96 – 1.14,RR = 1.05,0.294,Fail to reject H0
(b) Age <= 5,15.51%,14.63%,1.06,0.96 – 1.17,RR = 1.06,0.248,Fail to reject H0
"(c) Age <= 5, Low (<75k)",14.74%,16.30%,0.90,0.63 – 1.26,RR = 0.90,0.625,Fail to reject H0
"(c) Age <= 5, Medium (75-150k)",14.87%,13.77%,1.08,0.93 – 1.25,RR = 1.08,0.294,Fail to reject H0
"(c) Age <= 5, High (150k+)",16.23%,14.32%,1.13,0.92 – 1.40,RR = 1.13,0.239,Fail to reject H0


**Decision:** $p = 0.294$ (a), $0.248$ (b), and $> 0.23$ in every tier (c), so $p > 0.05 \implies$ **fail to reject $H_0$**.

**Takeaway:** BEVs do not claim more often than ICE cars, so no BEV frequency loading is needed.

## Test 1b: Third-Party Claim Frequency (Exact Poisson Rate Test)
$\lambda_{TP}$ = third-party claims per vehicle-year. A third-party claim is a collision with a third party and `TP_Damage_Amount > 0`, the same claims as in Test 4, so frequency × severity describe the same cover.
- $H_0$: $\lambda_{TP,\text{BEV}} = \lambda_{TP,\text{ICE}}$
- $H_1$: $\lambda_{TP,\text{BEV}} \neq \lambda_{TP,\text{ICE}}$

$Z = \log(\widehat{RR}) / \sqrt{1/x_{\text{BEV}} + 1/x_{\text{ICE}}}$ is the large-sample version of the test; the p-value is from the exact test.

In [5]:
# Third-party claim count per policy (0 if none); tp_data is defined in the setup cell
tp_freq_data <- policies |>
  left_join(count(tp_data, Policy_ID, name = "n_claims"), by = "Policy_ID") |>
  mutate(n_claims = coalesce(n_claims, 0L))

# Exact Poisson rate test with exposure, counts, rates and the log-scale Z statistic
tp_rate_test <- function(d) {
  s  <- d |> summarise(claims = sum(n_claims), exposure = sum(exposure), .by = Powertrain) |> arrange(Powertrain)
  pt <- poisson.test(s$claims, s$exposure)
  tibble(exposure_BEV = s$exposure[1], claims_BEV = s$claims[1], rate_BEV = s$claims[1] / s$exposure[1],
         exposure_ICE = s$exposure[2], claims_ICE = s$claims[2], rate_ICE = s$claims[2] / s$exposure[2],
         estimate = pt$estimate, ci_low = pt$conf.int[1], ci_high = pt$conf.int[2],
         z = log(pt$estimate) / sqrt(1 / s$claims[1] + 1 / s$claims[2]),
         statistic = paste0("RR = ", number(pt$estimate, 0.01)), p_value = pt$p.value)
}

t1b <- run(tp_freq_data, tp_rate_test)

t1b |>
  transmute(comparison,
            exposure_BEV = comma(exposure_BEV, 0.1), claims_BEV, rate_BEV = percent(rate_BEV, 0.01),
            exposure_ICE = comma(exposure_ICE, 0.1), claims_ICE, rate_ICE = percent(rate_ICE, 0.01),
            rr_95_ci = paste0(number(estimate, 0.001), " [", number(ci_low, 0.001), " – ", number(ci_high, 0.001), "]"),
            z = number(z, 0.001), p_value = fmt_p(p_value), decision)

comparison,exposure_BEV,claims_BEV,rate_BEV,exposure_ICE,claims_ICE,rate_ICE,rr_95_ci,z,p_value,decision
<chr>,<chr>,<int>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>
(a) All ages,"3,981.7",251,6.30%,"24,506.1",1432,5.84%,1.079 [0.940 – 1.234],1.108,0.276,Fail to reject H0
(b) Age <= 5,"3,624.2",233,6.43%,"10,044.7",556,5.54%,1.161 [0.992 – 1.356],1.918,0.058,Fail to reject H0


**Decision:** $p = 0.276$ (a) and $p = 0.058$ (b), so $p > 0.05 \implies$ **fail to reject $H_0$** in both. The age ≤ 5 result is borderline (RR = 1.16, $Z = 1.92$).

**Takeaway:** in a crude comparison, BEVs do not have a significantly higher third-party claim rate, but the point estimates are above 1 (1.08 and 1.16). Notebook 04 checks this with all rating factors held fixed.

## Test 2: Own-Damage Severity (F-test → Two-Sample t-test on log cost)
- F-test: $H_0$: $\sigma^2_{\text{BEV}} = \sigma^2_{\text{ICE}}$ vs. $H_1$: $\sigma^2_{\text{BEV}} \neq \sigma^2_{\text{ICE}}$
- t-test: $H_0$: $\mu_{\log(\text{OD}),\,\text{BEV}} = \mu_{\log(\text{OD}),\,\text{ICE}}$ vs. $H_1$: $\mu_{\log(\text{OD}),\,\text{BEV}} \neq \mu_{\log(\text{OD}),\,\text{ICE}}$
- (c): $H_0$: $\beta_{\text{BEV}} = 0$ in $\log(\text{OD}) = \beta_0 + \beta_{\text{BEV}} \mathbb{1}\{\text{BEV}\} + \gamma_{\text{tier}}$ (age ≤ 5)

`ratio` = geometric mean cost BEV / ICE.

In [6]:
fit <- sev_data |>
  filter(Vehicle_Age <= age_cutoff, !is.na(value_tier)) |>
  mutate(Powertrain = fct_relevel(Powertrain, "ICE")) |>
  lm(log(Own_Damage_Amount) ~ Powertrain + value_tier, data = _)
b  <- summary(fit)$coefficients["PowertrainBEV", ]

t2 <- bind_rows(
  run(sev_data, \(d) severity_test(d, Own_Damage_Amount)),
  tibble(comparison = "(c) Age <= 5, value tiers pooled", ratio = exp(b[["Estimate"]]),
         t = b[["t value"]], df = fit$df.residual, p_value = b[["Pr(>|t|)"]],
         statistic = paste0("t = ", number(t, 0.01), ", df = ", df), decision = decide(p_value)))
show_severity(t2)

comparison,F,p_F,ratio,t,df,p_value,decision
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
(a) All ages,1.01,0.860,1.98,13.02,4333,< 0.001,Reject H0
(b) Age <= 5,0.98,0.781,1.62,7.97,2027,< 0.001,Reject H0
"(c) Age <= 5, value tiers pooled",-,-,1.29,3.77,2024,< 0.001,Reject H0


**Decision:**
- F-test: $p_F = 0.860, 0.781 > 0.05 \implies$ equal variances, so the two-sample t-test applies.
- t-test: $p < 0.001 < 0.05$ in (a), (b) and (c) $\implies$ **reject $H_0$**.

**Takeaway:** BEV repairs cost more: 1.98× ICE on all ages, 1.62× at age ≤ 5 and 1.29× at equal age and value tier.

## Test 3: Total-Loss Rate (Two-Proportion Test + CMH)
$\pi$ = share of own-damage claims (excluding theft) that are a total loss.
- $H_0$: $\pi_{\text{BEV}} = \pi_{\text{ICE}}$
- $H_1$: $\pi_{\text{BEV}} \neq \pi_{\text{ICE}}$

(c) uses the Cochran-Mantel-Haenszel test over value tiers, because single tiers have too few total losses to test alone. Hypotheses: $H_0$: $\text{OR}_{\text{MH}} = 1$ vs. $H_1$: $\text{OR}_{\text{MH}} \neq 1$.

In [7]:
cmh_data <- tl_data |> filter(Vehicle_Age <= age_cutoff, !is.na(value_tier))
cmh <- mantelhaen.test(table(cmh_data$Powertrain, factor(cmh_data$is_total_loss_clean, levels = c(TRUE, FALSE)),
                             cmh_data$value_tier) * 1)            # * 1: doubles avoid integer overflow

t3 <- bind_rows(
  run(tl_data, prop_test),
  tibble(comparison = "(c) Age <= 5, value tiers (CMH)", estimate = unname(cmh$estimate),
         ci_low = cmh$conf.int[1], ci_high = cmh$conf.int[2],
         statistic = paste0("CMH X² = ", number(cmh$statistic, 0.01)),
         p_value = cmh$p.value, decision = decide(cmh$p.value)))
show(t3)

comparison,BEV,ICE,estimate,ci_95,statistic,p_value,decision
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
(a) All ages,8.6%,8.2%,0.41,-2.06 – 2.88,X² = 0.07,0.793,Fail to reject H0
(b) Age <= 5,7.7%,5.7%,2.07,-0.58 – 4.71,X² = 2.58,0.109,Fail to reject H0
"(c) Age <= 5, value tiers (CMH)",-,-,1.80,1.18 – 2.74,CMH X² = 7.45,0.006,Reject H0


**Decision:**
- (a) $p = 0.793$ and (b) $p = 0.109$, so $p > 0.05 \implies$ **fail to reject $H_0$**.
- (c) CMH: $p = 0.006 < 0.05 \implies$ **reject $H_0$**, with OR = 1.80.

**Takeaway:** BEVs are written off more often only once age and value are held fixed, and this rests on few cases.

## Test 4: Third-Party Damage Severity (F-test → Two-Sample t-test on log cost)
Sample: collisions with a third party and `TP_Damage_Amount > 0`.
- F-test: $H_0$: $\sigma^2_{\text{BEV}} = \sigma^2_{\text{ICE}}$ vs. $H_1$: $\sigma^2_{\text{BEV}} \neq \sigma^2_{\text{ICE}}$
- t-test: $H_0$: $\mu_{\log(\text{TP}),\,\text{BEV}} = \mu_{\log(\text{TP}),\,\text{ICE}}$ vs. $H_1$: $\mu_{\log(\text{TP}),\,\text{BEV}} \neq \mu_{\log(\text{TP}),\,\text{ICE}}$

In [8]:
t4 <- run(tp_data, \(d) severity_test(d, TP_Damage_Amount))
show_severity(t4)

comparison,F,p_F,ratio,t,df,p_value,decision
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
(a) All ages,0.95,0.608,0.95,-0.80,1681,0.423,Fail to reject H0
(b) Age <= 5,0.90,0.368,0.93,-0.92,787,0.360,Fail to reject H0


**Decision:**
- F-test: $p_F = 0.608, 0.368 > 0.05 \implies$ equal variances, so the two-sample t-test applies.
- t-test: $p = 0.423$ (a) and $0.360$ (b), so $p > 0.05 \implies$ **fail to reject $H_0$**.

**Takeaway:** BEVs do not cause more damage to other vehicles (ratio 0.95).

## Summary

In [9]:
summary_table <- bind_rows(
  `Test 1: Frequency`             = filter(t1, !str_starts(comparison, "\\(c\\)")) |> mutate(h0 = "λ_BEV = λ_ICE"),
  `Test 1b: TP frequency`        = mutate(t1b, h0 = "λ_TP,BEV = λ_TP,ICE"),
  `Test 2: Own-damage severity`   = mutate(t2, h0 = if_else(str_starts(comparison, "\\(c\\)"), "β_BEV = 0", "μ_log(OD),BEV = μ_log(OD),ICE")),
  `Test 3: Total loss`            = mutate(t3, h0 = if_else(str_starts(comparison, "\\(c\\)"), "OR_MH = 1", "π_BEV = π_ICE")),
  `Test 4: Third-party severity`  = mutate(t4, h0 = "μ_log(TP),BEV = μ_log(TP),ICE"),
  .id = "test") |>
  transmute(test, comparison, null_hypothesis = h0,
            statistic_p = paste0(statistic, "; p ", if_else(p_value < 0.001, "< 0.001", paste("=", number(p_value, 0.001)))),
            decision)

summary_table

test,comparison,null_hypothesis,statistic_p,decision
<chr>,<chr>,<chr>,<chr>,<chr>
Test 1: Frequency,(a) All ages,λ_BEV = λ_ICE,RR = 1.05; p = 0.294,Fail to reject H0
Test 1: Frequency,(b) Age <= 5,λ_BEV = λ_ICE,RR = 1.06; p = 0.248,Fail to reject H0
Test 1b: TP frequency,(a) All ages,"λ_TP,BEV = λ_TP,ICE",RR = 1.08; p = 0.276,Fail to reject H0
Test 1b: TP frequency,(b) Age <= 5,"λ_TP,BEV = λ_TP,ICE",RR = 1.16; p = 0.058,Fail to reject H0
Test 2: Own-damage severity,(a) All ages,"μ_log(OD),BEV = μ_log(OD),ICE","F = 1.01 (p = 0.860); t = 13.02, df = 4333; p < 0.001",Reject H0
Test 2: Own-damage severity,(b) Age <= 5,"μ_log(OD),BEV = μ_log(OD),ICE","F = 0.98 (p = 0.781); t = 7.97, df = 2027; p < 0.001",Reject H0
Test 2: Own-damage severity,"(c) Age <= 5, value tiers pooled",β_BEV = 0,"t = 3.77, df = 2024; p < 0.001",Reject H0
Test 3: Total loss,(a) All ages,π_BEV = π_ICE,X² = 0.07; p = 0.793,Fail to reject H0
Test 3: Total loss,(b) Age <= 5,π_BEV = π_ICE,X² = 2.58; p = 0.109,Fail to reject H0


**Overall:**
- **Frequency:** no difference.
- **Own-damage severity:** higher for BEV, about 1.3× at the same age and value.
- **Total loss:** higher for BEV only at the same age and value.
- **Third-party frequency:** no significant difference in the crude test (RR 1.08, p = 0.276; age ≤ 5: RR 1.16, p = 0.058).
- **Third-party damage:** no difference.